# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [84]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # TODO: completa a partir de la imagen
        self.start = (0, 0)
        self.wall = {(1, 1), (0,3), (2,4), (4,2)}
        self.slippery_states = { }

        self.terminal_states = {
            # (row, col): reward
            (2,2): +2,
            (0,5): +10,
            (3,5): -10,

        }

        self.danger_states = {
            # (row, col): -3
            (4,1): -3,
            (1,4): -3
        }
        self.living_reward = -1.0
        self.gamma = 0.9
        self.slippery_prob = 0.6

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width and state not in self.wall):
            return False
        return state != self.wall

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states 
        

    def get_reward(self, state):
        # TODO: implementa R(s)
        
        if self.is_terminal(state):
            return self.terminal_states[state]
        elif state in self.danger_states:
            return self.danger_states[state]
        else:
            return self.living_reward
            

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        perp1 = (action[1], action[0])  # perpendicular 1
        perp2 = (-action[1], -action[0])  # perpendicular 2
        if self.slippery_states.get(state, False):
            outcomes = [
                (action, self.slippery_prob),
                (perp1, (1 - self.slippery_prob) / 2),
                (perp2, (1 - self.slippery_prob) / 2)
            ]
        else:
            outcomes = [
                (action, 0.9),
                (perp1, 0.05),
                (perp2, 0.05)
            ]

        transitions = []
        for next_action, prob in outcomes:
            next_state = (state[0] + next_action[0], state[1] + next_action[1])
            if not self.is_valid_state(next_state):
                next_state = state
            transitions.append((next_state, prob))
        return transitions


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [85]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [86]:
def expected_next_value(grid, state, action, V):
    # TODO:
    # sum_{s'} T(s,a,s') V(s')
    return sum(prob * V[next_state] 
            for next_state, prob in grid.get_transition_probs(state, action))
    


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    # TODO
    # 1. Inicializa V(s) = 0 para todo s
    V = {state: 0.0 for state in grid.states()}
    deltas = []
    for i in range(max_iter):
        V_new = V.copy() 
        biggest_change = 0
        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                best_expected_value = max(
                    expected_next_value(grid, state, action, V) for action in grid.actions
                )
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * best_expected_value
                )
            biggest_change = max(biggest_change, abs(V_new[state] - V[state]))
        V = V_new
        deltas.append(biggest_change)
        if biggest_change < threshold:
            break
    return V, deltas
def extract_policy(grid, V):
    # TODO:
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue  
        else:
            best_action = max(
                grid.actions,
                key=lambda action: expected_next_value(grid, state, action, V)
            )
            policy[state] = best_action
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [87]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    # TODO
    V = {state: 0.0 for state in grid.states()}
    for i in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0
        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                action = policy[state]
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * expected_next_value(grid, state, action, V)
                )
            biggest_change = max(biggest_change, abs(V_new[state] - V[state]))
        V = V_new
        if biggest_change < threshold:
            break
    return V, i + 1


def policy_improvement(grid, V):
    # TODO
    new_policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue  
        else:
            best_action = max(
                grid.actions,
                key=lambda action: expected_next_value(grid, state, action, V)
            )
            new_policy[state] = best_action
    return new_policy   



def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # TODO:
    # 1. política inicial arbitraria
    initial_action = grid.actions[0]
    policy = {state: initial_action 
              for state in grid.states() if not grid.is_terminal(state)}
    # 2. evaluación
    # 3. mejora
    # 4. repetir hasta estabilidad
    history = []
    for i in range(max_iter):
        V, eval_iterations = policy_evaluation(grid, policy, threshold)
        history.append((policy.copy(), V.copy()))
        new_policy = policy_improvement(grid, V)
        changed = sum(new_policy[state] != policy[state] for state in new_policy)
        history.append({"policy_iteration": i+1, 
                        "evaluation_sweps": eval_iterations, 
                        "policy_changes": changed
                        })
        policy = new_policy
        if changed == 0:
            break
    V, _ = policy_evaluation(grid, policy, threshold)
    return policy, V, history



## Parte 4 — Visualización y comparación


In [88]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.wall:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.wall:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [89]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: [10.0, 7.920000000000001, 6.731100000000001, 1.9172902500000009, 0.7827765074999999, 0.3092794710750004, 0.12317586708712458, 0.09932973176186444, 0.06638332042868855, 0.05810248582313182, 0.048248607680425604, 0.027948171078377992, 0.014641190083926947, 0.005942930067248664, 0.002395064566881988, 0.0008620489793984554, 0.0003104069264967535, 0.00010541653350104596, 3.578999530429172e-05]

Valores:
 -2.407 |  -1.510 |  -0.462 |   WALL   |  +7.607 | +10.000
 -1.669 |   WALL   |  +0.774 |  +2.104 |  +3.670 |  +7.607
 -0.641 |  +0.624 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.581 |  -0.540 |  +0.579 |  -0.366 |  -1.469 | -10.000
 -2.564 |  -3.720 |   WALL   |  -1.469 |  -2.362 |  -3.522

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  ↑  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [({(0, 0): (-1, 0), (0, 1): (-1, 0), (0, 2): (-1, 0


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Respuestas
1. En ambos casos el robot prefiere la **entrega +10** partiendo desde `START`.
2. Esto se debe a que no estamos maximizando únicamente la recompensa final, sino la recompensa acumulada esperada y descontada en cada paso adicional por $$\gamma = 0.9$$ Como podemos observar, cada paso adicional que damos genera una recompensa de −living_reward. Además, para intentar llegar a la entrega +10, el robot debe recorrer un camino en el que acumula estos costos y está expuesto a estados con recompensas negativas, incluyendo los estados de peligro −3 y el riesgo de llegar al estado terminal −10. Por otro lado, la estación de carga +2 puede alcanzarse con un costo acumulado menor y, por lo tanto, su valor esperado resulta superior. Por esta razón, el modelo determina que la recompensa acumulada óptima se obtiene finalizando el recorrido en la estación de carga +2.
3. El piso resbaloso cambia la decisión en aquellos estados en los que la acción óptima de la política cambia al aumentar la probabilidad de desviación. Al pasar de una probabilidad de 0.90 de realizar correctamente la acción a 0.60, el riesgo asociado a ciertas acciones aumenta, por lo que el modelo puede preferir una acción alternativa que tenga una menor probabilidad de conducir a estados de recompensa negativa o terminales.
4. Este valor representa un costo grande para el robot al dar pasos, lo que hace que aunque existan estados con mayor recompensa, se premien estados con valor menor pero que estan más cercanos.
5. Por que existen dos tipos de estados para los que este valor es distinto, aquellos con piso normal y aquellos con piso resbaloso, por lo que, la transición debe implementarse como una función que considere tanto el estado actual como la acción, en lugar de utilizar una única distribución de probabilidades para todos los estados.
### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

En este caso no se penaliza de gran manera los recorridos largos, por lo que la politica debe presentar una mayor tendencia a llevar al estado con mayor recompensa **entrega +10**  

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


## Experimento A

In [90]:
# VALUE ITERATION
grid = WarehouseMDP()
grid.living_reward = -0.1
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: [10.0, 7.960500000000001, 6.8021775, 1.8443902500000005, 0.94160355525, 0.7220107835775, 0.5958414078890624, 0.5161349998447502, 0.42364100433982665, 0.25907286715964517, 0.19157142956022555, 0.1236116181607878, 0.05728780427841562, 0.03317450818953516, 0.012972136223089237, 0.006013489994052934, 0.0022216040528015846, 0.0009218641011523587, 0.000331631439022706, 0.00012903906225858464, 4.5728590347682285e-05]

Valores:
 +1.988 |  +2.375 |  +2.792 |   WALL   |  +8.591 | +10.000
 +1.660 |   WALL   |  +3.283 |  +3.911 |  +4.550 |  +8.591
 +1.400 |  +1.677 |  +2.000 |  +3.306 |   WALL   |  +7.537
 +1.481 |  +1.814 |  +2.356 |  +2.790 |  +2.356 | -10.000
 +1.078 |  -1.552 |   WALL   |  +2.356 |  +2.005 |  +1.125

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [({(0, 0): (-1, 0), (0, 1): (

## Experimento B


In [91]:
# VALUE ITERATION
grid = WarehouseMDP()
grid.slippery_prob = 0.4
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")

=== VALUE ITERATION ===
Iteraciones: [10.0, 7.920000000000001, 6.731100000000001, 1.9172902500000009, 0.7827765074999999, 0.3092794710750004, 0.12317586708712458, 0.09932973176186444, 0.06638332042868855, 0.05810248582313182, 0.048248607680425604, 0.027948171078377992, 0.014641190083926947, 0.005942930067248664, 0.002395064566881988, 0.0008620489793984554, 0.0003104069264967535, 0.00010541653350104596, 3.578999530429172e-05]

Valores:
 -2.407 |  -1.510 |  -0.462 |   WALL   |  +7.607 | +10.000
 -1.669 |   WALL   |  +0.774 |  +2.104 |  +3.670 |  +7.607
 -0.641 |  +0.624 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.581 |  -0.540 |  +0.579 |  -0.366 |  -1.469 | -10.000
 -2.564 |  -3.720 |   WALL   |  -1.469 |  -2.362 |  -3.522

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  ↑  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [({(0, 0): (-1, 0), (0, 1): (-1, 0), (0, 2): (-1, 0

## Experimento C

In [92]:
# VALUE ITERATION
grid = WarehouseMDP()
grid.gamma = 0.99
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")

=== VALUE ITERATION ===
Iteraciones: [10.0, 8.712, 8.144631, 3.1642976227500004, 1.1469491761241253, 0.7172607404376813, 0.648349250965995, 0.6297596759397928, 0.6140867655274398, 0.5749967538180991, 0.42875981056995816, 0.1961750143307457, 0.09667224461508672, 0.03833252896153949, 0.015991752690768912, 0.005976445125111862, 0.002295297128632079, 0.0008303307389092573, 0.00030463631044774786, 0.00010801350499356488, 3.857515528293831e-05]

Valores:
 -0.646 |  +0.518 |  +1.647 |   WALL   |  +8.601 | +10.000
 -1.533 |   WALL   |  +2.850 |  +4.118 |  +5.354 |  +8.601
 -0.428 |  +0.807 |  +2.000 |  +2.913 |   WALL   |  +7.395
 -1.425 |  -0.310 |  +0.849 |  +1.661 |  +0.474 | -10.000
 -2.574 |  -3.581 |   WALL   |  +0.474 |  -0.583 |  -2.119

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  ↑  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [({(0, 0): (-1, 0), (0, 1): (-1, 0), 

Podemos observar que, tomando $$\gamma = 0.99$$, efectivamente la politica óptima va desde el estado inicial hasta el estado terminal de **entrega +10**, esto es debido a que descuenta en menor medida los pasos adicionales que se dan en recorridos largos.
